# Customer & Marketing Analytics Team Assignment #2
> What Do Brands Actually Say - and Does It Work?\
> Analysing Sentiment, Topics and Emojis in Brand Social \Media Posts

> Group 10: Alice Williams, Lorenzo Zaniboni, Henry Soper, Alec Vitale, Anastasija Smiljkovska



In [ ]:
# import libraries
import io
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
import statsmodels.api as sm
import statsmodels.formula.api as smf
from google.colab import files
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
from tqdm import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Question 0 - Data Loading & Preparation

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name="Social Media Posts")
print(f"Success! Loaded {len(df)} rows.")

In [ ]:
# quick general check on the data

print(df.info()) # check for missing values and data types
print(f"\nBrand balance: \n{df['Brand'].value_counts()}\n") # confirm brand balance (should be 200 each)
print(f"Missing texts: {df['Post_Text'].isnull().sum()}\n") # check for empty posts
print(f"Preview text: \n{df['Post_Text'].head(10)}\n") # preview text to ensure emojis are rendering
display(df[['Num_Likes', 'Num_Comments']].describe()) # basic distribution of engagement

# Question 1 - Sentiment Analysis
How Positive Are the Posts?

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# get the continuous score (-1 to 1) for Q4 regression later
df['Sentiment_Score'] = df['Post_Text'].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])

# categorize for visualization
df['Sentiment_Label'] = pd.cut(df['Sentiment_Score'], bins=[-1, -0.05, 0.05, 1], labels=['Negative', 'Neutral', 'Positive'])

In [ ]:
# (a) summary statistics overall
print("--- OVERALL SENTIMENT STATISTICS ---")
display(df['Sentiment_Score'].describe())

# (b) sentiment by brand (find which is most positive/negative)
brand_comparison = df.groupby('Brand')['Sentiment_Score'].agg(['mean', 'median', 'std']).sort_values(by='mean', ascending=False)
print("\n--- SENTIMENT BY BRAND (Ranked Most Positive to Least) ---")
display(brand_comparison)

# (c) label counts (positive vs neutral vs negative)
print("\n--- SENTIMENT LABEL BREAKDOWN ---")
display(df.groupby(['Brand', 'Sentiment_Label']).size().unstack())

In [ ]:
# examine 3 most positive posts
print("--- TOP 3 MOST POSITIVE POSTS ---")
display(df.nlargest(3, 'Sentiment_Score')[['Brand', 'Post_Text', 'Sentiment_Score']])

# examine 3 most negative posts
print("\n--- TOP 3 MOST NEGATIVE POSTS ---")
display(df.nsmallest(3, 'Sentiment_Score')[['Brand', 'Post_Text', 'Sentiment_Score']])

In [ ]:
plt.figure(figsize=(12, 5))

# histogram of all scores
plt.subplot(1, 2, 1)
sns.histplot(df['Sentiment_Score'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of Sentiment Scores (All Posts)')

# boxplot by brand
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Brand', y='Sentiment_Score', palette='Set3')
plt.title('Sentiment Score by Brand')

plt.tight_layout()
plt.show()

# Question 2 - Topic Classification
What Are Brands Posting About?

In [ ]:
classifier = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-small", device=0)
candidate_labels = ["Informative", "Entertaining", "Promotional"]

print("Classifying topics (this takes ~2 mins)...")
tqdm.pandas()
df['Topic_Class'] = df['Post_Text'].progress_apply(lambda x: classifier(str(x), candidate_labels)['labels'][0])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Q1 plot: sentiment by brand
sns.boxplot(data=df, x='Brand', y='Sentiment_Score', ax=ax1, palette='Set2')
ax1.set_title('Q1: Sentiment Distribution by Brand')

# Q2 plot: topic mix by brand
df.groupby('Brand')['Topic_Class'].value_counts(normalize=True).unstack().plot(kind='bar', stacked=True, ax=ax2)
ax2.set_title('Q2: Content Strategy (Topic Mix) per Brand')
ax2.set_ylabel('Proportion of Posts')

plt.tight_layout()
plt.show()

# quick check of the results
display(df[['Brand', 'Post_Text', 'Sentiment_Score', 'Topic_Class']].head())

# Question 3 - Paralinguistic / Emoji Analysis
The Role of Emojis

In [ ]:
# function to extract emojis from text
def extract_emojis(text):
    return ''.join(c for c in str(text) if c in emoji.EMOJI_DATA)

df['Emojis'] = df['Post_Text'].apply(extract_emojis)

In [ ]:
# get top 10 emojis overall
all_emojis = "".join(df['Emojis'])
top_10_overall = Counter(all_emojis).most_common(10)

print("--- TOP 10 EMOJIS OVERALL ---")
for emo, count in top_10_overall:
    print(f"{emo} : {count}")

In [ ]:
# emoji usage by brand (frequency)
print("\n--- EMOJI FREQUENCY BY BRAND ---")
df['Emoji_Count'] = df['Emojis'].apply(len)
brand_emoji_stats = df.groupby('Brand')['Emoji_Count'].mean().sort_values(ascending=False)
print(brand_emoji_stats)

In [ ]:
# top emoji per brand
print("\n--- SIGNATURE EMOJI PER BRAND ---")
for brand in df['Brand'].unique():
    brand_text = "".join(df[df['Brand'] == brand]['Emojis'])
    top_emo = Counter(brand_text).most_common(1)
    print(f"{brand}: {top_emo[0][0]} (used {top_emo[0][1]} times)")

# Question 4 - Drivers of Engagement
What Makes Posts Perform?

In [ ]:
# create dummy variables for Topic_Class, use 'Promotional' as the baseline to compare against
df_final = pd.get_dummies(df, columns=['Topic_Class'], drop_first=False)

# model for Likes
formula_likes = 'Num_Likes ~ Sentiment_Score + Emoji_Count + Topic_Class_Informative + Topic_Class_Entertaining + Total_Followers + Total_Posts_LastYear'
model_likes = smf.ols(formula=formula_likes, data=df_final).fit()

# model for Comments
formula_comments = 'Num_Comments ~ Sentiment_Score + Emoji_Count + Topic_Class_Informative + Topic_Class_Entertaining + Total_Followers + Total_Posts_LastYear'
model_comments = smf.ols(formula=formula_comments, data=df_final).fit()

print("--- REGRESSION RESULTS: LIKES ---")
print(model_likes.summary())

print("\n\n--- REGRESSION RESULTS: COMMENTS ---")
print(model_comments.summary())

# Question 5 - Brand Differences
Do Engagement Drivers Differ Across Brands?

In [ ]:
# loop through each brand to compare them
brands = df['Brand'].unique()

for brand in brands:
    print(f"\n" + "="*50)
    print(f"RESULTS FOR: {brand}")
    print("="*50)

    brand_df = df_final[df_final['Brand'] == brand]

    # run Likes model for this specific brand
    model_l = smf.ols(formula=formula_likes, data=brand_df).fit()

    # run Comments model for this specific brand
    model_c = smf.ols(formula=formula_comments, data=brand_df).fit()

    print(f"--- LIKES (Predictors for {brand}) ---")
    display(model_l.summary2().tables[1][['Coef.', 'P>|t|']])

    print(f"--- COMMENTS (Predictors for {brand}) ---")
    display(model_c.summary2().tables[1][['Coef.', 'P>|t|']])